In [1]:
import re
from pathlib import Path

import numpy as np
import pandas as pd

# Input and output paths stay relative so the notebook works from the repo root.
source_path = Path("assets/raw_iot_data.csv")
transformed_path = Path("assets/iot_data_transformed.csv")
hourly_summary_path = Path("assets/iot_data_hourly_summary.csv")

# Load the raw dataset.
df = pd.read_csv(source_path)
df.columns = [column.strip() for column in df.columns]

print(f"Loaded {len(df)} rows from {source_path}")
print("Raw columns:", list(df.columns))

# Standardize obvious missing values and parse timestamps.
working = df.copy()
for col in working.columns:
    mask = working[col].isin(["", " "])
    working.loc[mask, col] = pd.NA

working["timestamp"] = pd.to_datetime(working["timestamp"], errors="coerce")
working["hour"] = working["timestamp"].dt.floor("h")

# Keep reference to raw data value
working["data_value_raw"] = working["data_value"]

# Vectorized extraction of numeric and unit_text
s_str = working["data_value"].astype(str).str.strip()
is_empty = working["data_value"].isna() | (s_str == "") | (s_str == "nan") | (s_str == "<NA>")

# Extract number and suffix
extracted = s_str.str.extract(r"(-?\d+(?:\.\d+)?)(.*)")

numeric_value = pd.Series(np.nan, index=working.index)
unit_text = pd.Series(pd.NA, index=working.index, dtype="object")

# No match (but not empty)
no_match = extracted[0].isna() & ~is_empty
unit_text = np.where(no_match, s_str, unit_text)

# Match rows
matched = extracted[0].notna() & ~is_empty
numeric_value = np.where(matched, extracted[0].astype(float), numeric_value)

suffix = extracted[1].str.strip()
suffix_clean = np.where(suffix == "", pd.NA, suffix)
unit_text = np.where(matched, suffix_clean, unit_text)

working["numeric_value"] = numeric_value
working["unit_text"] = pd.Series(unit_text, index=working.index).replace({np.nan: pd.NA})

# Identify missing values before imputing.
missing_summary = working.isna().sum().sort_values(ascending=False)
print("\nMissing values by column:")
print(missing_summary[missing_summary > 0].to_string())

# Flag duplicate rows
duplicate_mask = working.duplicated(
    subset=["timestamp", "device_id", "data_type", "data_value_raw"],
    keep="first",
)
working["is_duplicate"] = duplicate_mask

# Vectorized inconsistent check
expected_units = {
    "Temperature": {"°C", "C", "degC"},
    "Humidity": {"%", "% humidity", "humidity", "percent"},
    "Energy": {"kWh", "wh", "Wh", "KWh"},
}
expected_units_lower = {k: {u.lower() for u in v} for k, v in expected_units.items()}
is_inconsistent = pd.Series(False, index=working.index)

for data_type, allowed_units in expected_units_lower.items():
    mask = (working["data_type"] == data_type) & working["unit_text"].notna()
    if not mask.any():
        continue
    unit_lower = working.loc[mask, "unit_text"].astype(str).str.strip().str.lower()
    is_inconsistent.loc[mask] = ~unit_lower.isin(allowed_units)

working["is_inconsistent"] = is_inconsistent

# Vectorized outlier detection
counts = working.groupby("data_type")["numeric_value"].transform("count")
q1 = working.groupby("data_type")["numeric_value"].transform("quantile", 0.25)
q3 = working.groupby("data_type")["numeric_value"].transform("quantile", 0.75)
iqr = q3 - q1
lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr

working["is_outlier"] = (counts >= 4) & (
    (working["numeric_value"] < lower_bound)
    | (working["numeric_value"] > upper_bound)
)

# Vectorized median fill
medians = working.groupby("data_type")["numeric_value"].transform("median")
overall_median = working["numeric_value"].median()
working["numeric_value_filled"] = working["numeric_value"].fillna(medians).fillna(overall_median)

# Cleaned dataset (excluding duplicates)
cleaned_df = working.loc[~working["is_duplicate"]].copy()

# Vectorized hourly aggregate
cleaned_df_temp = cleaned_df.assign(is_missing=cleaned_df["data_value_raw"].isna())
hourly_summary = (
    cleaned_df_temp.groupby(["hour", "data_type"], dropna=False)
    .agg(
        reading_count=("numeric_value_filled", "count"),
        average_value=("numeric_value_filled", "mean"),
        median_value=("numeric_value_filled", "median"),
        minimum_value=("numeric_value_filled", "min"),
        maximum_value=("numeric_value_filled", "max"),
        missing_value_count=("is_missing", "sum"),
        inconsistency_count=("is_inconsistent", "sum"),
        outlier_count=("is_outlier", "sum"),
    )
    .reset_index()
    .sort_values(["hour", "data_type"])
)

# Persist the transformed outputs for downstream analysis.
cleaned_df.to_csv(transformed_path, index=False)
hourly_summary.to_csv(hourly_summary_path, index=False)

print("\nData quality checks:")
print(f"Exact duplicates removed: {int(duplicate_mask.sum())}")
print(f"Inconsistent rows flagged: {int(working['is_inconsistent'].sum())}")
print(f"Outliers flagged: {int(working['is_outlier'].sum())}")
print(f"Transformed detail rows saved: {len(cleaned_df)}")
print(f"Hourly summary rows saved: {len(hourly_summary)}")
print(f"Saved cleaned dataset to: {transformed_path}")
print(f"Saved hourly summary to: {hourly_summary_path}")

print("\nCleaned data preview:")
display(cleaned_df.head())
print("\nHourly summary preview:")
display(hourly_summary.head())

Loaded 10 rows from assets/raw_iot_data.csv
Raw columns: ['timestamp', 'device_id', 'data_type', 'data_value']

Missing values by column:
data_value        2
data_value_raw    2
numeric_value     2
unit_text         2

Data quality checks:
Exact duplicates removed: 0
Inconsistent rows flagged: 4
Outliers flagged: 0
Transformed detail rows saved: 10
Hourly summary rows saved: 10
Saved cleaned dataset to: assets/iot_data_transformed.csv
Saved hourly summary to: assets/iot_data_hourly_summary.csv

Cleaned data preview:


,timestamp,device_id,data_type,data_value,hour,data_value_raw,numeric_value,unit_text,is_duplicate,is_inconsistent,is_outlier,numeric_value_filled
0,2025-03-04 20:41:46.466097,Device_4,Energy,22.5°C,2025-03-04 20:00:00,22.5°C,22.5,°C,False,True,False,22.5
1,2025-03-04 21:41:46.466095,Device_5,Humidity,45%,2025-03-04 21:00:00,45%,45.0,%,False,False,False,45.0
2,2025-03-04 22:41:46.466093,Device_3,Temperature,18.3 kWh,2025-03-04 22:00:00,18.3 kWh,18.3,kWh,False,True,False,18.3
3,2025-03-04 23:41:46.466090,Device_5,Humidity,NaN,2025-03-04 23:00:00,NaN,NaN,NaN,False,False,False,45.0
4,2025-03-05 00:41:46.466089,Device_5,Humidity,23.1°C,2025-03-05 00:00:00,23.1°C,23.1,°C,False,True,False,23.1



Hourly summary preview:


,hour,data_type,reading_count,average_value,median_value,minimum_value,maximum_value,missing_value_count,inconsistency_count,outlier_count
0,2025-03-04 20:00:00,Energy,1,22.5,22.5,22.5,22.5,0,1,0
1,2025-03-04 21:00:00,Humidity,1,45.0,45.0,45.0,45.0,0,0,0
2,2025-03-04 22:00:00,Temperature,1,18.3,18.3,18.3,18.3,0,1,0
3,2025-03-04 23:00:00,Humidity,1,45.0,45.0,45.0,45.0,1,0,0
4,2025-03-05 00:00:00,Humidity,1,23.1,23.1,23.1,23.1,0,1,0
